# TB 1AG: Colab Improvement Notebook

This notebook runs the 7-year crypto pipeline, compares baseline vs tuned runs, and helps iterate toward better out-of-sample performance.

In [ ]:
#@title Clone repository (run once per fresh Colab runtime)
import os

REPO_URL = "https://github.com/comprono/vapar.git"
REPO_DIR = "/content/vapar"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}


In [ ]:
#@title Install dependencies
!python -m pip install --upgrade pip
!pip install -r backend/requirements.txt


In [ ]:
#@title Baseline run (current repo defaults)
import sys
sys.path.insert(0, "/content/vapar")

from research_layer.crypto_autoresearch import run_crypto_7y_autoresearch

baseline_report = run_crypto_7y_autoresearch()
print("Baseline report:", baseline_report["report_path"])
print("Baseline summary:", baseline_report["summary"])


In [ ]:
#@title Tuned experiments (objective: reduce gap vs buy-and-hold)
import importlib
import json
import pandas as pd

import research_layer.crypto_autoresearch as ca

EXPERIMENTS = [
    {
        "name": "fast_stride1",
        "AUTORESEARCH_MONTH_STRIDE": 1,
        "DEFAULT_CONFIGS": [
            {"hidden_layers": [64, 32], "alpha": 0.0005, "learning_rate_init": 0.0008, "threshold": 0.0015, "max_iter": 220, "trade_mode": "long_short", "model_kind": "regression"},
            {"hidden_layers": [96, 48], "alpha": 0.0010, "learning_rate_init": 0.0007, "threshold": 0.0500, "max_iter": 220, "trade_mode": "long_only", "model_kind": "classification"},
            {"hidden_layers": [128, 64], "alpha": 0.0010, "learning_rate_init": 0.0006, "threshold": 0.0600, "max_iter": 240, "trade_mode": "long_short", "model_kind": "classification"}
        ]
    },
    {
        "name": "conservative_costaware",
        "AUTORESEARCH_MONTH_STRIDE": 1,
        "DEFAULT_CONFIGS": [
            {"hidden_layers": [64, 32], "alpha": 0.0010, "learning_rate_init": 0.0007, "threshold": 0.0030, "max_iter": 200, "trade_mode": "long_only", "model_kind": "regression"},
            {"hidden_layers": [96, 48], "alpha": 0.0015, "learning_rate_init": 0.0006, "threshold": 0.0700, "max_iter": 220, "trade_mode": "long_only", "model_kind": "classification"},
            {"hidden_layers": [96, 48], "alpha": 0.0010, "learning_rate_init": 0.0006, "threshold": 0.0800, "max_iter": 220, "trade_mode": "long_short", "model_kind": "classification"}
        ]
    }
]

rows = []
reports = {}

for exp in EXPERIMENTS:
    ca = importlib.reload(ca)
    ca.AUTORESEARCH_MONTH_STRIDE = exp["AUTORESEARCH_MONTH_STRIDE"]
    ca.DEFAULT_CONFIGS = exp["DEFAULT_CONFIGS"]

    report = ca.run_crypto_7y_autoresearch()
    reports[exp["name"]] = report
    s = report["summary"]
    rows.append({
        "experiment": exp["name"],
        "avg_model_total_return": s.get("avg_model_total_return"),
        "avg_buyhold_total_return": s.get("avg_buyhold_total_return"),
        "avg_excess_vs_buyhold": s.get("avg_excess_vs_buyhold"),
        "avg_model_sharpe": s.get("avg_model_sharpe"),
        "symbols_accepted": s.get("symbols_accepted"),
        "acceptance_rate": s.get("acceptance_rate"),
        "report_path": report.get("report_path")
    })

results_df = pd.DataFrame(rows).sort_values(
    ["avg_excess_vs_buyhold", "avg_model_sharpe", "acceptance_rate"],
    ascending=False
)
results_df


In [ ]:
#@title Inspect best experiment per-coin details
best_name = results_df.iloc[0]["experiment"]
best_report = reports[best_name]

print("Best experiment:", best_name)
print("Best config:", best_report["autorresearch"]["best_config"])

coin_rows = []
for item in best_report["results_by_symbol"]:
    coin_rows.append({
        "symbol": item["symbol"],
        "model_total_return": item["model_walkforward"]["total_return"],
        "buyhold_total_return": item["buy_and_hold"]["total_return"],
        "excess_vs_buyhold": item["model_walkforward"]["total_return"] - item["buy_and_hold"]["total_return"],
        "model_sharpe": item["model_walkforward"]["sharpe"],
        "accepted": item["acceptance"]["accepted"],
        "verification_csv": item["daily_verification_path"]
    })

pd.DataFrame(coin_rows).sort_values("excess_vs_buyhold", ascending=False)


In [ ]:
#@title Save best config artifact for later code integration
best_cfg = best_report["autorresearch"]["best_config"]
artifact = {
    "picked_experiment": best_name,
    "best_config": best_cfg,
    "summary": best_report["summary"],
    "report_path": best_report["report_path"]
}

out_path = "/content/vapar/data/reports/colab_best_config.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print("Saved:", out_path)
artifact


## Notes

- Keep anti-cheat rule: oracle must remain verification-only.
- Prioritize `avg_excess_vs_buyhold` and `symbols_accepted`.
- If all experiments fail acceptance, adjust model family or feature set (not only thresholds).